In [ ]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

# 1. Setup Dummy Data
data = {
    'person_id': ['A', 'A', 'A', 'A', 'B', 'B'],
    'month_reference': pd.to_datetime(['2023-01-01', '2023-02-01', '2023-03-01', '2023-04-01', '2023-01-01', '2023-02-01']),
    'date_overdue': pd.to_datetime(['2023-01-10', '2023-02-10', '2023-03-10', '2023-04-10', '2023-01-15', '2023-02-15']).date,
    'date_paid': ['2023-01-08', '2023-02-12', None, '2023-04-09', '2023-01-15', '2023-02-20'],
    'invoice_value': [100.0, 200.0, 300.0, 400.0, 50.0, None]
}
df_sklearn = pd.DataFrame(data)

# 2. Custom Transformer
class TimeSeriesFeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # Nothing to learn/fit for these specific operations
        return self

    def transform(self, X):
        # Create a copy to avoid SettingWithCopy warnings on the original DF
        X_out = X.copy()
        
        # --- A. Cleaning & Type Casting ---
        # Equivalent to SQL CAST(date_paid AS DATE)
        X_out['date_paid_cleaned'] = pd.to_datetime(X_out['date_paid']).dt.date
        
        # Equivalent to SQL COALESCE / PySpark fillna
        X_out['invoice_value_cleaned'] = X_out['invoice_value'].fillna(0.0)
        
        # --- B. Feature Engineering ---
        # Equivalent to SQL DATEDIFF
        # Note: We must ensure columns are proper datetime types for subtraction
        X_out['days_to_pay'] = (
            pd.to_datetime(X_out['date_paid_cleaned']) - 
            pd.to_datetime(X_out['date_overdue'])
        ).dt.days

        # --- C. Rolling Aggregation (The Complex Part) ---
        # 1. Sort is critical! SQL and Spark do this via "ORDER BY" in window
        X_out = X_out.sort_values(by=['person_id', 'month_reference'])
        
        # 2. Groupby + Rolling
        # equivalent to: PARTITION BY person_id ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        X_out['rolling_avg_3m_invoice'] = (
            X_out.groupby('person_id')['invoice_value_cleaned']
            .rolling(window=3, min_periods=1) # min_periods=1 mimics SQL behavior for first rows
            .mean()
            .reset_index(0, drop=True) # Reset index to align back to DataFrame
        )
        
        return X_out

# 3. Create and Run Pipeline
pipeline = Pipeline(steps=[
    ('feature_engineering', TimeSeriesFeatureEngineer())
])

# 4. Execute
result_df = pipeline.fit_transform(df_sklearn)

# 5. Display
print(result_df[['person_id', 'month_reference', 'invoice_value', 'rolling_avg_3m_invoice', 'days_to_pay']])

  person_id month_reference  invoice_value  rolling_avg_3m_invoice  \
0         A      2023-01-01          100.0                   100.0   
1         A      2023-02-01          200.0                   150.0   
2         A      2023-03-01          300.0                   200.0   
3         A      2023-04-01          400.0                   300.0   
4         B      2023-01-01           50.0                    50.0   
5         B      2023-02-01            NaN                    25.0   

   days_to_pay  
0         -2.0  
1          2.0  
2          NaN  
3         -1.0  
4          0.0  
5          5.0  
